In [11]:
import os
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display


In [5]:
MODELllama = 'llama3.2'
MODELPhi = 'phi3'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

#if not running, run "ollama serve" at a command line

In [ ]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    message = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model = MODELllama, messages = message)
    return response.choices[0].message.content

In [ ]:
# This can reveal the "training cut off", or the most recent date in the training data

display(Markdown(message_gpt("what is the weather today?")))

I'm a large language model, I don't have real-time access to current weather information. My training data only goes up until December 2023.

However, I can suggest ways for you to find out the current weather:

1. Check online weather websites: You can check websites like AccuWeather, Weather.com, or the National Weather Service (NWS) for current weather conditions and forecasts.
2. Use a mobile app: Many weather apps, such as Dark Sky or Weather Underground, provide real-time weather information and forecasts for your location.
3. Tune into local news: You can also check local news broadcasts or online news websites for weather updates.

If you give me the name of your city or location, I'd be happy to help you find out the general climate and average weather patterns for that area!

## Gradio UI

In [16]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [17]:
shout("Hello")

Shout has been called with input Hello


'HELLO'

In [ ]:
gr.Interface(fn = shout, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello
Shout has been called with input whatS up
Shout has been called with input hello! this is Pradeep


<h2 style="color:#dbc900;"> Using Gradio's Share tool</h2>
            <span style="color:#dbc900;">There is a way to share Gradio UI with others. This deploys the gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This uses an advanced technology known as 'HTTP tunneling' (like ngrok for people who know it) which isn't allowed by many Antivirus programs and corporate environments.<br/>
            </span>

In [ ]:
# Adding share=True means that it can be accessed publically.
# A more permanent hosting is available using a platform called Spaces from HuggingFace.
# link will be generated which will be valid for a week. you need to click on the link to see in web browser.
# Adding inbrowser=True opens up a new browser window automatically : .launch(inbrowser=True)

gr.Interface(fn = shout, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch(share = True)

## Adding Authentication

Gradio makes it very easy to have userids and passwords

In [ ]:
gr.Interface(fn = shout, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch(share = True, inbrowser = True, auth = ("pradeep", "123"))

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# creating gradio ui

message_input = gr.Textbox(label = "Enter message: ", info = "Enter a message to be shouted", lines = 7)
message_output = gr.Textbox(label = "Respinse", lines = 8)

view = gr.Interface(
    fn = shout,
    title = "Shout",
    inputs = [message_input],
    outputs = [message_output],
    examples = ["hello, My name is Pradeep Kumar", "Please print these words in capital letters"],
    flagging_mode = "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Please print these words in capital letters
Shout has been called with input hello, My name is Pradeep Kumar


In [30]:
# creating gradio ui for message_gpt

message_input = gr.Textbox(label = "Enter message: ", info = "Enter a message for message_gpt", lines = 7)
message_output = gr.Textbox(label = "Respinse", lines = 8)

view = gr.Interface(
    fn = message_gpt,
    title = "message_gpt",
    inputs = [message_input],
    outputs = [message_output],
    examples = ["hello, My name is Pradeep Kumar", "Please print these words in capital letters"],
    flagging_mode = "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [31]:
# Let's use Markdown
# Are you wondering why it makes any difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global variable, used back in the message_gpt function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for message_gpt", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="message_gpt", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [56]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),

def stream_gpt(prompt):
    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = openai.chat.completions.create(model = MODELllama, messages = message, stream = True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [33]:
# Let's use Markdown
# Are you wondering why it makes any difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global variable, used back in the message_gpt function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for message_gpt", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="message_gpt", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello! my name is Pradeep
Shout has been called with input hello! my name is Pradeep


And now getting fancy

In [54]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),

def stream_phi(prompt):
    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = openai.chat.completions.create(model = MODELPhi, messages = message, stream = True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [53]:
def stream_model(prompt, model):
    if model=="Model llama3.2":
        result = stream_gpt(prompt)
    elif model=="Model Phi3":
        result = stream_phi(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result ## shortcut for : for chunk in result: yield chunk

In [58]:
message_input = gr.Textbox(label = "Your Message", info = "Enter a message for LLM", lines = 7)
message_output = gr.Markdown(label = "Response")
model_selector = gr.Dropdown(["Model llama3.2", "Model Phi3"], label = "Select Model", value = "Model llama3.2") #value is default model selected

view = gr.Interface(
    fn = stream_model,
    title = "LLMs",
    inputs = [message_input, model_selector],
    outputs = [message_output],
    examples=[
            ["Explain the Transformer architecture to a layperson in 100 words", "Model llama3.2"],
            ["Explain the Transformer architecture to an aspiring AI engineer in 100 words", "Model Phi3"]
        ], 
    flagging_mode = "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
